# OPT-13B FP16 WikiText-2 Baseline (layer-wise, L4)

Evaluate the public model `facebook/opt-13b` on the WikiText-2 test split with FP16 weights and activations. No quantization is used.

Formal evaluation settings: **non-overlapping 2048-token blocks; the incomplete final block is dropped**. This matches the protocol used by the other baseline notebooks in this project.

The FP16 weights (~25.7 GB) do not fit in an L4 (24 GB). The model is therefore kept in CPU RAM and evaluated **layer by layer**: the hidden states of every 2048-token block stay on the GPU, and one decoder layer at a time is moved to the GPU, applied to all blocks, and moved back. Each block still goes through exactly the same modules with the same inputs as a full forward pass, so the loss is identical (verified bit-exact against `model(...).loss` on random-initialized OPT models, including OPT-13B dimensions). Expected peak GPU memory is about 5 GiB.

Requirements: an NVIDIA CUDA GPU and at least 40 GiB of system RAM (Colab L4 provides about 53 GB). The model is public, so no Hugging Face token is required.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import json
import math
import platform
import time
from pathlib import Path

import datasets
import psutil
import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "facebook/opt-13b"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
EXECUTION_MODE = "layerwise_cpu_to_gpu_streaming"
MIN_SYSTEM_RAM_GIB = 40
OUTPUT_PATH = Path("opt-13b-baseline-s2048.json")

if not torch.cuda.is_available():
    raise RuntimeError("This baseline requires an NVIDIA CUDA GPU.")
system_ram_gib = psutil.virtual_memory().total / 2**30
if system_ram_gib < MIN_SYSTEM_RAM_GIB:
    raise RuntimeError(f"OPT-13B needs >= {MIN_SYSTEM_RAM_GIB} GiB system RAM; found {system_ram_gib:.1f} GiB.")

DEVICE = torch.device("cuda:0")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"Model: {MODEL_ID}")
print(f"Protocol: {EVALUATION_PROTOCOL}")
print(f"Execution: {EXECUTION_MODE}")
print(f"System RAM: {system_ram_gib:.1f} GiB")
print(f"Output: {OUTPUT_PATH}")

## Load the original FP16 model into CPU RAM

OPT-13B is public and can be loaded directly. `use_fast=False` is retained as a compatibility hint, but Transformers v5 may still select its unified fast backend. The actual tokenizer class and `is_fast` value are recorded instead of requiring a specific backend. `device_map="cpu"` keeps every weight in CPU RAM; the evaluation below moves modules to the GPU explicitly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="cpu",
    attn_implementation="eager",
)
model.eval()
model.config.use_cache = False

parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
parameter_devices = {(p.device.type, p.device.index) for p in model.parameters()}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {("cpu", None)}, parameter_devices
assert not getattr(model, "is_quantized", False)

max_positions = int(model.config.max_position_embeddings)
if CONTEXT_LENGTH > max_positions:
    raise ValueError(
        f"CONTEXT_LENGTH={CONTEXT_LENGTH} exceeds max_position_embeddings={max_positions}."
    )

print(f"Tokenizer: {tokenizer.__class__.__name__}, is_fast={tokenizer.is_fast}")
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Maximum context: {max_positions:,}")

## Prepare WikiText-2

The text joining and evaluation protocol are kept identical to the other model baselines. Token counts may differ because OPT uses a different tokenizer.

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
class _StopForward(Exception):
    pass


def _opt_pre_modules(model):
    decoder = model.model.decoder
    return [m for m in (decoder.embed_tokens, decoder.embed_positions, decoder.project_in) if m is not None]


def _opt_post_modules(model):
    decoder = model.model.decoder
    return [m for m in (decoder.final_layer_norm, decoder.project_out) if m is not None]


def _move(modules, device):
    for module in modules:
        module.to(device)


def _same_kwargs(reference, candidate):
    if reference.keys() != candidate.keys():
        return False
    for key, value in reference.items():
        other = candidate[key]
        if torch.is_tensor(value):
            if not (torch.is_tensor(other) and value.shape == other.shape and torch.equal(value, other)):
                return False
        elif value != other:
            return False
    return True


@torch.inference_mode()
def capture_first_layer_inputs(model, blocks, device):
    """Run embeddings on GPU and stop at decoder layer 0, collecting its inputs."""
    layers = model.model.decoder.layers
    captured = {}

    def stop_hook(module, args, kwargs):
        captured["hidden_states"] = args[0] if args else kwargs["hidden_states"]
        captured["kwargs"] = {k: v for k, v in kwargs.items() if k != "hidden_states"}
        raise _StopForward

    pre_modules = _opt_pre_modules(model)
    _move(pre_modules, device)
    handle = layers[0].register_forward_pre_hook(stop_hook, with_kwargs=True)
    hidden = None
    layer_kwargs = None
    try:
        for index, batch in enumerate(blocks):
            try:
                model(batch.to(device), use_cache=False)
            except _StopForward:
                pass
            else:
                raise RuntimeError("Decoder layer 0 was not reached.")
            states = captured["hidden_states"]
            if hidden is None:
                hidden = torch.empty((len(blocks), *states.shape[1:]), dtype=states.dtype, device=device)
                layer_kwargs = captured["kwargs"]
            elif not _same_kwargs(layer_kwargs, captured["kwargs"]):
                raise RuntimeError(f"Decoder layer kwargs differ at block {index}.")
            hidden[index] = states[0]
    finally:
        handle.remove()
        _move(pre_modules, "cpu")
    return hidden, layer_kwargs


@torch.inference_mode()
def run_decoder_layers(model, hidden, layer_kwargs, device):
    """Stream one decoder layer at a time to GPU and update all blocks in place."""
    layers = model.model.decoder.layers
    for layer in tqdm(layers, desc="Decoder layers"):
        layer.to(device)
        for index in range(hidden.size(0)):
            output = layer(hidden[index:index + 1], **layer_kwargs)
            hidden[index:index + 1] = output[0] if isinstance(output, tuple) else output
        layer.to("cpu")
    return hidden


@torch.inference_mode()
def block_losses(model, hidden, blocks, device):
    """Apply the final norm/projection and lm_head, returning the HF causal-LM loss of every block."""
    post_modules = _opt_post_modules(model) + [model.lm_head]
    _move(post_modules, device)
    losses = []
    try:
        for index, batch in enumerate(blocks):
            states = hidden[index:index + 1]
            for module in post_modules[:-1]:
                states = module(states)
            logits = model.lm_head(states).contiguous()
            labels = batch.to(device)
            loss = model.loss_function(logits=logits, labels=labels, vocab_size=model.config.vocab_size)
            losses.append(loss.float().item())
    finally:
        _move(post_modules, "cpu")
    return losses


def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = DEVICE
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_blocks = usable_length // context_length
    blocks = [input_ids[:, begin:begin + context_length] for begin in range(0, usable_length, stride)]
    assert len(blocks) == total_blocks

    torch.cuda.synchronize(device)  # Initializes CUDA; the model itself starts on CPU.
    torch.cuda.reset_peak_memory_stats(device)
    start_time = time.perf_counter()

    hidden, layer_kwargs = capture_first_layer_inputs(model, blocks, device)
    run_decoder_layers(model, hidden, layer_kwargs, device)
    losses = block_losses(model, hidden, blocks, device)
    del hidden

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time

    total_nll = 0.0
    total_loss_tokens = 0
    for index, loss in enumerate(losses):
        if not math.isfinite(loss):
            raise FloatingPointError(f"Non-finite FP16 loss at block {index}.")
        loss_tokens = context_length - 1
        total_nll += loss * loss_tokens
        total_loss_tokens += loss_tokens
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

In [ ]:
metrics = evaluate_perplexity(model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER)
result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "dtype": "float16",
    "quantized": False,
    "attention_implementation": "eager",
    "execution_mode": EXECUTION_MODE,
    "system_ram_gib": system_ram_gib,
    "tokenizer_use_fast": tokenizer.is_fast,
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "model_max_position_embeddings": max_positions,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **metrics,
}

print(json.dumps(result, indent=2, ensure_ascii=False))
OUTPUT_PATH.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))